# SAC Collector — D4RL Tier Audit

**目的**：在 Plan A（`agent_step_*.pt` 训练过程切片做真 D4RL tier）开跑前盘点 Drive 上的 ckpt 文件，回答 3 个决策性问题：

| 问题 | 答案需要什么 |
|---|---|
| **Q1.** `cross_u10_regression/.../seed_46/` 真的存了 `agent_step_*.pt` 吗？多少个？ | `ls *.pt` + 文件大小 |
| **Q2.** 这些 step ckpt 的 SACConfig schema 与本地一致（`obs_dim=48`、`privileged_obs_dim=0`）？ | `torch.load` × 4 后比 config |
| **Q3.** 它们的 success rate 谱是否真是 `random → expert` 单调？（D4RL tier 划分的物理基础） | 30 ep deterministic eval × 4 step ckpt |

**预算**：~15 min L4（其中 30-ep eval × 4 ckpt ≈ 10 min CPU）。

**参考文档**：
- [`docs/arrival_v2_sac_collector_design.md`](../docs/arrival_v2_sac_collector_design.md) rev.2 §4.2 / §5.1
- 2026-05-24 session 决策：放弃路径 1（cross_u15 sensor-floored），改用同 seed 训练切片做真 D4RL tier

## 0. 环境检查

In [ ]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name:    {torch.cuda.get_device_name(0)}")

## 1. 挂载 Drive + cd 到项目根

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd && git log --oneline -1

## 2. Q1 — 盘点 cross_u10 seed_46 的所有 ckpt 文件

**期望**：`agent_step_100000.pt` ~ `agent_step_1000000.pt`（每 100k 一个，共 10 个）+ `agent_best.pt` / `agent_final.pt` / `agent_latest.pt`。

**失败模式**：
- 只有 `agent_best/final/latest.pt` 3 个文件 → `checkpoint_every_steps` 当时被设成 1M（或更大），**Plan A 不可行，需 fallback Plan B 或 Plan C**
- 部分 step ckpt 缺失 → Drive sync 中断或被清理过

In [ ]:
from pathlib import Path

CKPT_DIR = Path("checkpoints/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46")
EXP_DIR  = Path("experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46")

print(f"CKPT_DIR exists: {CKPT_DIR.exists()}")
print(f"EXP_DIR  exists: {EXP_DIR.exists()}")
print()
if CKPT_DIR.exists():
    files = sorted(CKPT_DIR.glob("*.pt"))
    print(f"Found {len(files)} .pt files:")
    for f in files:
        print(f"  {f.name:35s}  {f.stat().st_size/1e6:6.1f} MB")
else:
    print("⚠️ CKPT_DIR not found — Drive 上路径与 spec §2.5 不一致，检查 prototype 分支 sync 状态")

## 3. 读 trainer_state.json — 确认 `checkpoint_every_steps` 实际值

In [ ]:
import json

ts_path = EXP_DIR / "trainer_state.json"
if ts_path.exists():
    ts = json.loads(ts_path.read_text())
    tc = ts.get("train_config", {})
    print(f"total_env_steps:         {ts.get('env_step')}")
    print(f"checkpoint_every_steps:  {tc.get('checkpoint_every_steps')}")
    print(f"random_steps:            {tc.get('random_steps')}")
    print(f"update_after:            {tc.get('update_after')}")
    print(f"num_envs:                {tc.get('num_envs')}")
    print(f"history_length:          {ts.get('history_length')}")
    print(f"probe_layout:            {ts.get('probe_layout')}")
    print(f"reward_objective:        {ts.get('reward_objective')}")
    print(f"algorithm:               {ts.get('algorithm')}")
    print(f"best_agent_path:         {ts.get('best_agent_path')}")
    print(f"final_agent_path:        {ts.get('final_agent_path')}")
else:
    print(f"⚠️ {ts_path} not found")

## 4. Q2 — schema 一致性

加载 4 个候选 step ckpt（覆盖 random / medium / medium-expert / expert 4 档），验证：
1. 文件存在
2. `payload["config"]` 是合法 SACConfig dict
3. obs_dim / action_dim / privileged_obs_dim 跨 step **完全一致**（schema 没漂移）

**候选 step 选取原则**：早、中、中后、末 4 档；如果 SAC 真在 cross_u10 是平滑训练，应能拿到 random→expert 谱。

In [ ]:
# 候选 step。可按 §2 输出实际存在的 step 调整。
CANDIDATE_STEPS = [100_000, 300_000, 600_000, 1_000_000]

step_payloads = {}
for step in CANDIDATE_STEPS:
    path = CKPT_DIR / f"agent_step_{step}.pt"
    if not path.exists():
        print(f"step {step:>8}: ❌ NOT FOUND")
        continue
    payload = torch.load(str(path), map_location="cpu", weights_only=False)
    cfg = payload.get("config", {})
    step_payloads[step] = (path, cfg)
    print(f"step {step:>8}: ✅  obs_dim={cfg.get('obs_dim')}  "
          f"action_dim={cfg.get('action_dim')}  "
          f"priv_dim={cfg.get('privileged_obs_dim')}  "
          f"hidden={cfg.get('hidden_dim')}  "
          f"ln={cfg.get('use_layernorm')}")

if step_payloads:
    obs_dims = {step: cfg.get("obs_dim") for step, (_, cfg) in step_payloads.items()}
    if len(set(obs_dims.values())) == 1:
        print(f"\n✅ schema 一致：所有 step ckpt obs_dim = {next(iter(obs_dims.values()))}")
    else:
        print(f"\n⚠️ schema 漂移：{obs_dims}")

## 5. Q3 — 30-ep deterministic eval：success rate 谱

对每个候选 step ckpt 跑 **30 episode deterministic eval**（task 随机化，每 ep 重新采 init pose + flow phase），目标是看 success rate 是否真是 random→expert 单调爬升。

**期望谱（猜测）**：
- step 100k：~0.1-0.2（actor 刚学会 goal-seek 大方向）
- step 300k：~0.5（medium）
- step 600k：~0.85（medium-expert）
- step 1000k：~1.0（expert，与 spec §2.5 final=1.0 对齐）

**预算**：30 ep × 4 ckpt × ~5s/ep on CPU ≈ 10 min。

In [ ]:
from types import SimpleNamespace
import numpy as np
import time

from auv_nav.env import ObservationHistoryWrapper
from auv_nav.sac_policy import SACCheckpointPolicy
from scripts.train_utils import (
    make_env_config_overrides,
    make_planar_env,
    make_reset_options,
)

# 与 ckpt 训练 reset_options 严格一致（来自 trainer_state.json）
EVAL_FLOW = "wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy"
EVAL_EPISODES = 30
EVAL_ARGS = SimpleNamespace(
    probe_layout="s0",
    history_length=4,
    task_geometry="cross_stream",
    target_speed=1.5,
    difficulty=None,
    action_mode=None,
    speed_ratio=None,
    objective="arrival_v2",
    energy_cost_gain=None,
    safety_cost_gain=None,
)
ENV_OVERRIDES = make_env_config_overrides(EVAL_ARGS)
RESET_OPTIONS = make_reset_options(EVAL_ARGS)


def eval_ckpt(ckpt_path: Path, n_eps: int = EVAL_EPISODES) -> dict:
    policy = SACCheckpointPolicy.from_checkpoint(
        ckpt_path, device="cpu", deterministic=True
    )
    env = make_planar_env(
        EVAL_FLOW,
        history_length=EVAL_ARGS.history_length,
        probe_layout=EVAL_ARGS.probe_layout,
        env_config_overrides=ENV_OVERRIDES,
    )
    try:
        base_env = env.env if isinstance(env, ObservationHistoryWrapper) else env
        successes, oob, timeout, lengths = 0, 0, 0, []
        for ep in range(n_eps):
            obs, info = env.reset(seed=ep, options=RESET_OPTIONS)
            done = False
            ep_len = 0
            while not done:
                action = policy.act(base_env, obs)
                obs, _r, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                ep_len += 1
            reason = str(info.get("reason", ""))
            if info.get("success", False):
                successes += 1
            if reason == "out_of_bounds":
                oob += 1
            elif reason == "timeout":
                timeout += 1
            lengths.append(ep_len)
        return {
            "success": successes / n_eps,
            "oob": oob / n_eps,
            "timeout": timeout / n_eps,
            "mean_len": float(np.mean(lengths)),
        }
    finally:
        env.close()

results = {}
t0 = time.time()
for step, (path, _) in step_payloads.items():
    t_step = time.time()
    r = eval_ckpt(path)
    results[step] = r
    print(f"step {step:>8}: success={r['success']:.2%}  oob={r['oob']:.2%}  "
          f"timeout={r['timeout']:.2%}  mean_len={r['mean_len']:.0f}  "
          f"({time.time()-t_step:.1f}s)")
print(f"\nTotal eval time: {time.time()-t0:.1f}s")

## 6. Plan A 可行性判定

**判定规则**：

| 判定 | 条件 | 行动 |
|---|---|---|
| ✅ **PLAN A GO** | step ckpt ≥ 3 个存在 + success 谱单调（最低 ≤ 0.3 / 最高 ≥ 0.85） | 直接进 4 个 dataset 收集（spec §4.2 修订版） |
| ⚠️ **PLAN A PARTIAL** | step ckpt 存在但 success 谱不单调 / tier 划分不清 | 重选 step（看具体谱再定），可能用 `agent_step_{200k, 400k, 700k, 1000k}` 等 |
| ❌ **PLAN A NO-GO** | step ckpt 不存在 / 只有 best+final+latest 3 个 | 回退 Plan B（replay_latest.pkl → npz）或 Plan C（纯 expert tier） |

In [ ]:
if not results:
    verdict = "❌ NO-GO — step ckpt 未找到，需 fallback"
else:
    successes_sorted = [results[s]["success"] for s in sorted(results.keys())]
    is_monotone = all(
        successes_sorted[i] <= successes_sorted[i + 1] + 0.1  # 容忍 ±10pp 抖动
        for i in range(len(successes_sorted) - 1)
    )
    span_ok = max(successes_sorted) - min(successes_sorted) >= 0.5
    expert_ok = max(successes_sorted) >= 0.85
    if is_monotone and span_ok and expert_ok:
        verdict = f"✅ GO — success 谱 {[f'{s:.0%}' for s in successes_sorted]}，可直接进 Plan A 数据收集"
    elif span_ok and expert_ok:
        verdict = f"⚠️ PARTIAL — span/expert OK 但单调性弱，需重选 step：{[f'{s:.0%}' for s in successes_sorted]}"
    else:
        verdict = f"❌ NO-GO — success 谱不够分散或 expert 不达 85%：{[f'{s:.0%}' for s in successes_sorted]}"
print(verdict)

## 7. Fallback 路径准备：盘点 `replay_latest.pkl`（Plan B 备份）

如果 Plan A NO-GO，需要 fallback Plan B（`replay_latest.pkl` → npz 转 medium-replay）。这一节只做盘点（不转），让用户看到 pickle 是否就在 Drive 上。

In [ ]:
replay_path = EXP_DIR / "state" / "replay_latest.pkl"
if replay_path.exists():
    size_mb = replay_path.stat().st_size / 1e6
    print(f"✅ {replay_path}")
    print(f"   size: {size_mb:.1f} MB ({size_mb/1024:.2f} GB)")
    print(f"   → 若 Plan A NO-GO，可走 Plan B：写 scripts/replay_to_offline.py 转 npz")
else:
    print(f"⚠️ {replay_path} not found — Plan B 也不可用，只能走 Plan C（pure expert tier）")

## 8. 决策模板（手填后回传 main session）

请把下面这段填好后贴回 main session，我会根据它决定下一步：

```
### SAC Collector D4RL Tier Audit — 回传

Q1 (step ckpt 存在性):
  - 找到 N 个 agent_step_*.pt（具体 step 列表：____）
  - 文件大小是否一致：是 / 否

Q2 (schema 一致性):
  - obs_dim 跨 step：（应为 48）____
  - privileged_obs_dim：（应为 0）____
  - 一致性：✅ / ⚠️

Q3 (success rate 谱):
  step 100k:  ____  (oob ____, timeout ____)
  step 300k:  ____
  step 600k:  ____
  step 1000k: ____

判定: ✅ GO / ⚠️ PARTIAL / ❌ NO-GO
verdict 行内容：____

replay_latest.pkl 存在: 是 / 否（size ____）
```